In [ ]:
res_csv = f'{CACHE_DIR}/results.csv'
if os.path.exists(res_csv):
    results = pd.read_csv(res_csv)
else:
    results = pd.DataFrame(columns=['tag', 'sub', 'top1', 'top5', 'median_rank'])

for s in DEV_SUBS:
    truth = load_test_labels(s)
    r = ranked_ids(fused[s])
    rank = (r == truth[:, None]).argmax(1) + 1
    row = {'tag': 'fused_enc+dec', 'sub': s,
           'top1': float((rank <= 1).mean()),
           'top5': float((rank <= 5).mean()),
           'median_rank': float(np.median(rank))}
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)

summary = results.groupby('tag').agg(
    top1_mean=('top1', 'mean'), top1_sd=('top1', 'std'),
    top5_mean=('top5', 'mean'), top5_sd=('top5', 'std'),
    rank_med =('median_rank', 'median'),
    n=('top1', 'count')).sort_values('top1_mean', ascending=False)
summary.round(3).to_csv(f'{FIG_DIR}/results_summary.csv')
print(summary.round(3))

## Results table for the report

Pull the running tally from `cache_04/results.csv` (04's ablations) and prepend the fused-ensemble row for completeness.

In [ ]:
def load_fused():
    enc = {s: np.load(f'{SCORES_DIR}/encoding_sub-{s:02d}.npz')['S'] for s in SUBS}
    dec = {s: np.load(f'{SCORES_DIR}/decoder_sub-{s:02d}.npz' )['S'] for s in SUBS}
    def rk(S): return 1 - np.argsort(np.argsort(-S, 1), 1) / (S.shape[1]-1)
    return {s: 0.5 * rk(enc[s]) + 0.5 * rk(dec[s]) for s in SUBS}
# Keep this independent of 06's chosen w — just a diagnostic.
fused = load_fused()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, s in zip(axes, DEV_SUBS):
    truth = load_test_labels(s)
    pred  = ranked_ids(fused[s])[:, 0]
    M = np.zeros((102, 102))
    truth_idx = truth - 1001
    pred_idx  = pred  - 1001
    for ti, pi in zip(truth_idx, pred_idx):
        M[ti, pi] += 1
    ax.imshow(M, cmap='magma')
    ax.set_title(f'sub-{s:02d}  diag={int(np.trace(M))}/{len(truth)}')
    ax.set_xlabel('predicted id − 1000'); ax.set_ylabel('true id − 1000')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/confusion_matrix.pdf')
plt.show()

print('\nWorst-ranked recordings (fused rank of true id):')
for s in DEV_SUBS:
    truth = load_test_labels(s)
    r = ranked_ids(fused[s])
    rank = (r == truth[:, None]).argmax(1) + 1
    worst = np.argsort(-rank)[:3]
    for idx in worst:
        print(f'  sub-{s:02d}  recording {idx:3d}  true={truth[idx]}  rank={int(rank[idx])}  predicted={r[idx, 0]}')

## 3. Confusion and failure cases

Fused-stream top-1 confusion matrix per subject (sub-01..04) and the three worst-ranked test recordings (highest true-id rank).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
rows = ['u_V', 'u_S', 'u_A', 'sh_VS', 'sh_VA', 'sh_SA']
labels = ['vision unique', 'semantic unique', 'audio unique',
          'vision ∩ semantic', 'vision ∩ audio', 'semantic ∩ audio']
subs = list(part_df.index)
x = np.arange(len(subs))
bottom = np.zeros(len(subs))
for r, lbl in zip(rows, labels):
    vals = np.maximum(part_df[r].values.astype(float), 0)
    ax.bar(x, vals, bottom=bottom, label=lbl)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels([f'sub-{s:02d}' for s in subs])
ax.set_ylabel('CV R² (stacked)')
ax.legend(fontsize=8, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/variance_partitioning.pdf')
plt.show()

In [ ]:
GROUPS = {
    'vision':   ['alexnet_features.0', 'alexnet_features.3', 'alexnet_features.6',
                 'alexnet_features.8', 'alexnet_features.10', 'videomae_v2'],
    'semantic': ['clip_vitl14', 'dinov2_vitl14'],
    'audio':    ['clap', 'lowlevel'],
}

def concat_banks(banks, vids, var=0.95):
    Xs = []
    for b in banks:
        if not os.path.exists(f'{FEAT_DIR}/{b}.npz'): continue
        X = stack_bank(b, vids)
        sc = StandardScaler().fit(X); X = sc.transform(X)
        pca = PCA(n_components=var, svd_solver='full').fit(X); X = pca.transform(X)
        Xs.append(StandardScaler().fit_transform(X))
    return np.concatenate(Xs, axis=1).astype(np.float32)

def cv_r2(X, Y, n_folds=5):
    """Mean per-sample R² across (ch, t)."""
    n, ch, t = Y.shape
    Yflat = Y.reshape(n, -1)
    pred = np.zeros_like(Yflat)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    for tr, va in kf.split(X):
        m = RidgeCV(alphas=np.logspace(-3, 6, 20)).fit(X[tr], Yflat[tr])
        pred[va] = m.predict(X[va])
    ss_res = ((Yflat - pred) ** 2).sum(0)
    ss_tot = ((Yflat - Yflat.mean(0)) ** 2).sum(0) + 1e-9
    return 1 - ss_res.mean() / ss_tot.mean()

def partition_for_subject(sub):
    ids, Y = load_pp(sub)
    sets = {k: concat_banks(v, ids) for k, v in GROUPS.items()}
    # R² for every union
    def fit(subset):
        Xs = [sets[k] for k in subset]
        X = np.concatenate(Xs, axis=1) if Xs else None
        if X is None: return 0.
        return cv_r2(X, Y)
    V, S, A      = fit(['vision']), fit(['semantic']), fit(['audio'])
    VS           = fit(['vision','semantic'])
    VA           = fit(['vision','audio'])
    SA           = fit(['semantic','audio'])
    VSA          = fit(['vision','semantic','audio'])
    # inclusion-exclusion
    u_V = VSA - SA
    u_S = VSA - VA
    u_A = VSA - VS
    sh_VS = (V + S - VS)
    sh_VA = (V + A - VA)
    sh_SA = (S + A - SA)
    return dict(V=V, S=S, A=A, VSA=VSA, u_V=u_V, u_S=u_S, u_A=u_A,
                sh_VS=sh_VS, sh_VA=sh_VA, sh_SA=sh_SA)

part = {s: partition_for_subject(s) for s in DEV_SUBS}
part_df = pd.DataFrame(part).T
print(part_df.round(3))

## 2. Variance partitioning (vision / semantic / audio)

Define three groups:
- **vision**: AlexNet (5 layers) + VideoMAE
- **semantic**: CLIP + DINOv2
- **audio**: CLAP + lowlevel audio channels

For each group, pooled CV R² is computed on sub-01..04. Unique and shared variance uses inclusion-exclusion over the three combined fits — the standard variance-partitioning approach.

In [ ]:
times = np.arange(N_TIME) / SFREQ_D
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, s in zip(axes.flat, DEV_SUBS):
    for b, r in curves[s].items():
        ax.plot(times, r, label=b, lw=1)
    ax.set_title(f'sub-{s:02d}'); ax.axhline(0, c='gray', lw=0.5)
    ax.set_xlabel('time from onset (s)')
axes[0, 0].legend(fontsize=7, ncol=2); axes[0, 0].set_ylabel('mean channel r')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/encoding_curves.pdf')
plt.show()

In [ ]:
def stack_bank(name, vids):
    z = np.load(f'{FEAT_DIR}/{name}.npz')
    return np.stack([z[f'{int(v):04d}'] for v in vids]).astype(np.float32)

def load_pp(sub):
    z = np.load(f'{CACHE_DIR}/pp_sub-{sub:02d}.npz')
    return z['ids'], z['Y_trn']

def bank_cv_pred(X, Y, n_folds=5, var=0.95):
    """OOF ridge predictions. X: (n, d), Y: (n, ch, t)."""
    n, ch, t = Y.shape
    Yflat = Y.reshape(n, -1)
    pred  = np.zeros_like(Yflat)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    for tr, va in kf.split(X):
        sc = StandardScaler().fit(X[tr])
        Xt = sc.transform(X[tr]); Xv = sc.transform(X[va])
        pca = PCA(n_components=var, svd_solver='full').fit(Xt)
        Xt = pca.transform(Xt);    Xv = pca.transform(Xv)
        m = RidgeCV(alphas=np.logspace(-3, 6, 20)).fit(Xt, Yflat[tr])
        pred[va] = m.predict(Xv)
    return pred.reshape(n, ch, t)

def per_time_r(Y_true, Y_pred):
    a = Y_true - Y_true.mean(0, keepdims=True)
    b = Y_pred - Y_pred.mean(0, keepdims=True)
    r = (a * b).mean(0) / (a.std(0) * b.std(0) + 1e-6)
    return r.mean(0)                                       # (t,)

BANKS = ['lowlevel',
         'alexnet_features.0', 'alexnet_features.3', 'alexnet_features.6',
         'alexnet_features.8', 'alexnet_features.10',
         'clip_vitl14', 'dinov2_vitl14', 'videomae_v2', 'clap']

def encoding_curves(sub):
    ids, Y = load_pp(sub)
    out = {}
    for b in BANKS:
        if not os.path.exists(f'{FEAT_DIR}/{b}.npz'): continue
        X = stack_bank(b, ids)
        Yp = bank_cv_pred(X, Y)
        out[b] = per_time_r(Y, Yp)
    return out

curves = {s: encoding_curves(s) for s in DEV_SUBS}

## 1. Layer × latency encoding curves

For each feature bank, fit a single-bank ridge via 5-fold CV and compute per-sample Pearson r between predicted and true EEG, averaged across channels. One line per bank; one panel per subject.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

TRAIN_DIR     = './trn'
TEST_DIR      = './tst'
TEST_STIM_DIR = './tst_stim'
FEAT_DIR      = './features'
CACHE_DIR     = './cache_04'
SCORES_DIR    = './scores'
FIG_DIR       = './figs'
os.makedirs(FIG_DIR, exist_ok=True)

SFREQ_D   = 100
N_CHAN    = 17
N_TIME    = 300
TEST_IDS  = np.arange(1001, 1103)
DEV_SUBS  = [1, 2, 3, 4]
SUBS      = [1, 2, 3, 4, 5, 6]

def load_test_labels(sub):
    return np.load(f'{TEST_STIM_DIR}/sub-{sub:02d}_tst_stim.npy').astype(int)

def ranked_ids(S, ids=TEST_IDS):
    return np.asarray(ids)[np.argsort(-S, axis=1)]

# 07 — Analysis plots for the report

Takes the per-subject preprocessed EEG (`cache_04/pp_sub-XX.npz`), the banded-ridge predictions (`cache_04/br_sub-XX.npz`), and the fused scores (recomputed from `scores/`) and produces:

1. **Layer × latency encoding curves** — per-feature-bank CV R² heatmap over EEG time, averaged across channels.
2. **Variance partitioning** — vision vs. semantic vs. audio contribution to CV R² (unique + shared variance).
3. **Confusion and failure cases** — confusion matrix of top-1 picks on sub-01..04 and the three worst-ranked recordings per subject.

Figures are written to `figs/` as PDFs/PNGs for the report.